In [1]:
import zipfile
import os

zip_path = "credit.zip"  
extract_path = "data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))

Extracted files: ['credit.lisp', 'credit.names', 'crx.data', 'crx.names', 'Index', 'personal_expense_classification.csv']


In [2]:
import pandas as pd  


columns = [
    "A1","A2","A3","A4","A5","A6","A7","A8",
    "A9","A10","A11","A12","A13","A14","A15","Target"
]

# Loading the dataset
df = pd.read_csv("data/crx.data", header=None, names=columns)


df.head()


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,Target
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,00202,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,00043,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,00280,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,00100,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,00120,0,+


In [4]:
df["Target"].value_counts()


-    383
+    307
Name: Target, dtype: int64

In [3]:
# Replacing '?' with NaN
df.replace("?", pd.NA, inplace=True)

# Checking missing values
df.isnull().sum()


A1        12
A2        12
A3         0
A4         6
A5         6
A6         9
A7         9
A8         0
A9         0
A10        0
A11        0
A12        0
A13        0
A14       13
A15        0
Target     0
dtype: int64

In [4]:
# Identifing categorical columns
categorical_cols = df.select_dtypes(include="object").columns

# Fill missing categorical values with mode
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Identifing numerical columns
numerical_cols = df.select_dtypes(exclude="object").columns

# Fill missing numerical values with median
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])


In [6]:
X = df.drop("Target", axis=1)  # input features
y = df["Target"]               # output


In [7]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    criterion="entropy", 
    max_depth=4           
)

model.fit(X, y)


DecisionTreeClassifier(criterion='entropy', max_depth=4)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy_score(y_test, y_pred)


0.8333333333333334